# Rules 模块教程

本教程详细介绍 open-xquant 的 **Rules 模块**——策略管道的最后一公里。

在 `engine_module` 教程中，我们学习了 Engine 的四阶段管道：Universe → Indicator → Signal → **Rule**。当时只用到了最基础的 `EntryRule` 和 `ExitRule`。实际上，open-xquant 的 Rule 层远比这丰富——它包含 **5 类规则**，在每个 bar 上按固定顺序执行：

```
Stage 1: Risk Rules       → 组合级熔断（可冻结后续所有阶段）
Stage 2a: Process Orders  → SimBroker 检查挂单是否触发（不受冻结影响）
Stage 2b: Order Rules     → 提交止损/止盈/追踪止损挂单
Stage 3: Rebalance Rules  → 按权重再平衡
Stage 4: Exit Rules       → 卖出信号触发平仓
Stage 5: Entry Rules      → 买入信号触发建仓
```

当 Risk Rules 返回 `hold=True` 时，Stage 2b~5 全部跳过，但 Stage 2a（已挂的止损/止盈单）仍然会正常执行。

本教程通过 7 个独立章节，逐一演示每类 Rule 的 API、行为和组合用法。

## 0. 准备工作

### 安装

```bash
pip install open-xquant[yfinance]
```

### 下载数据

本教程使用 AAPL 2023~2024 年的历史数据：

In [1]:
from oxq.data import YFinanceDownloader

downloader = YFinanceDownloader()
downloader.download("AAPL", start="2023-01-01", end="2024-12-31")
print("数据下载完成")

数据下载完成


### 公共配置

所有章节共享同一套 SMA 金叉/死叉信号作为基础管道。后续章节只替换 Rule 部分，保持其他组件不变，以便公平对比不同 Rule 的效果。

In [2]:
from decimal import Decimal

from oxq.core import Engine, Strategy
from oxq.data import LocalMarketDataProvider
from oxq.indicators import SMA
from oxq.signals import Crossover
from oxq.trade import SimBroker
from oxq.universe import StaticUniverse

SYMBOL = "AAPL"
START = "2023-01-01"
END = "2024-12-31"
INITIAL_CASH = 100_000.0

# 所有章节共享的管道组件
COMMON_INDICATORS = {
    "sma_10": (SMA(), {"column": "close", "period": 10}),
    "sma_50": (SMA(), {"column": "close", "period": 50}),
}

COMMON_SIGNALS = {
    "golden_cross": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
}


def run(strategy, **broker_kwargs):
    """运行策略并返回结果的快捷函数。"""
    broker = SimBroker(**broker_kwargs)
    return Engine().run(
        strategy,
        market=LocalMarketDataProvider(),
        router=broker,
        receiver=broker,
        start=START, end=END,
        initial_cash=INITIAL_CASH,
    )


def compare(results: dict, extra_rows=None):
    """打印多策略对比表。"""
    header = f"{'':>14}" + "".join(f"{label:>16}" for label in results)
    print(header)
    print("-" * len(header))
    rows = [
        ("总收益率", lambda r: f"{r.total_return():.2%}"),
        ("Sharpe", lambda r: f"{r.sharpe_ratio():.2f}"),
        ("最大回撤", lambda r: f"{r.max_drawdown():.2%}"),
        ("交易次数", lambda r: f"{len(r.trades)}"),
        ("期末资产", lambda r: f"{r.equity_curve[-1][1]:,.0f}"),
    ]
    if extra_rows:
        rows.extend(extra_rows)
    for name, fn in rows:
        vals = "".join(f"{fn(r):>16}" for r in results.values())
        print(f"{name:>14}{vals}")


def show_trades(result, max_rows=10):
    """打印交易记录。"""
    if not result.trades:
        print("  (无交易)")
        return
    print(f"  {'日期':<28} {'方向':>4} {'数量':>6} {'订单类型':<14} {'成交价':>10} {'手续费':>8}")
    print("  " + "-" * 76)
    for fill in result.trades[:max_rows]:
        o = fill.order
        print(
            f"  {fill.filled_at:<28} {o.side:>4} {o.shares:>6} "
            f"{o.order_type:<14} {fill.filled_price:>10.2f} {fill.fee:>8.2f}"
        )
    if len(result.trades) > max_rows:
        print(f"  ... 共 {len(result.trades)} 笔，仅显示前 {max_rows} 笔")


print(f"公共配置: {SYMBOL}, {START}~{END}, 初始资金 {INITIAL_CASH:,.0f}")

公共配置: AAPL, 2023-01-01~2024-12-31, 初始资金 100,000


---
## 1. Entry Rules — 买入规则

Entry Rules 在 Stage 5（最后）执行，负责在信号触发时建仓。open-xquant 提供 4 种 Entry Rule，覆盖从简单到精细的仓位控制需求：

| 规则 | 买入逻辑 | 适用场景 |
|------|----------|----------|
| `EntryRule` | 固定股数 | 快速验证信号 |
| `TargetValueEntryRule` | 按目标市值计算股数 | 等权配置 |
| `FullPositionEntryRule` | 用全部现金买入 | 集中投资 |
| `SizedEntryRule` | 固定股数 + 仓位约束 | 生产级策略 |

**共同行为**：所有 Entry Rule 都在信号列为 `True` 且当前**无持仓**时触发。已有持仓的 symbol 不会重复建仓。

### 1.1 EntryRule — 固定股数

最简单的买入规则。每次信号触发时买入固定数量的股票，不考虑股价和资金量。

适合策略研发初期，快速验证信号是否有效。

In [3]:
import pandas as pd
from oxq.core import Portfolio, Position
from oxq.rules import EntryRule

entry = EntryRule(signal="golden_cross", shares=100)

# 场景 1: 信号触发 + 无持仓 → 买入
row = pd.Series({"close": 180.0, "golden_cross": True})
order = entry.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"信号触发 + 无持仓: {order}")

# 场景 2: 信号触发 + 已有持仓 → 不重复买入
portfolio_with_pos = Portfolio(
    cash=Decimal("80000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("170"))},
)
order2 = entry.evaluate("AAPL", row, portfolio_with_pos)
print(f"信号触发 + 已有持仓: {order2}")

# 场景 3: 信号未触发 → 不买入
row_no_signal = pd.Series({"close": 180.0, "golden_cross": False})
order3 = entry.evaluate("AAPL", row_no_signal, Portfolio(cash=Decimal("100000")))
print(f"信号未触发: {order3}")

信号触发 + 无持仓: Order(symbol='AAPL', side='BUY', shares=100, order_type='market', limit_price=None, stop_price=None, trail_pct=None)
信号触发 + 已有持仓: None
信号未触发: None


### 1.2 TargetValueEntryRule — 按目标市值

计算「要达到目标市值还需买多少股」，适合等权配置多个标的。

例如：目标市值 50,000，当前股价 200 → 需要 250 股。如果已持有 100 股，只补买 150 股。

In [4]:
from oxq.rules import TargetValueEntryRule

rule_tv = TargetValueEntryRule(signal="golden_cross", target_value=50_000)
row = pd.Series({"close": 200.0, "golden_cross": True})

# 无持仓: 50000 / 200 = 250 股
order = rule_tv.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"无持仓 → 买入 {order.shares} 股 (目标 250 股)")

# 已有 100 股: 补买 250 - 100 = 150 股
order2 = rule_tv.evaluate("AAPL", row, Portfolio(
    cash=Decimal("80000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("180"))},
))
print(f"已有 100 股 → 补买 {order2.shares} 股")

无持仓 → 买入 250 股 (目标 250 股)
已有 100 股 → 补买 150 股


### 1.3 FullPositionEntryRule — 全仓买入

用全部可用现金买入，资金利用率最高但风险集中。适合单标的策略或高确信度信号。

In [5]:
from oxq.rules import FullPositionEntryRule

rule_fp = FullPositionEntryRule(signal="golden_cross")
row = pd.Series({"close": 200.0, "golden_cross": True})

# 10 万现金: 100000 / 200 = 500 股
order = rule_fp.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"10 万现金 → 买入 {order.shares} 股")

# 3 万现金: 30000 / 200 = 150 股
order2 = rule_fp.evaluate("AAPL", row, Portfolio(cash=Decimal("30000")))
print(f"3 万现金 → 买入 {order2.shares} 股")

10 万现金 → 买入 500 股
3 万现金 → 买入 150 股


### 1.4 SizedEntryRule — 带仓位约束的买入

在固定股数基础上增加两个可选的仓位控制参数：

- **`max_position`**: 限制该标的的总持仓股数上限
- **`max_pct_equity`**: 限制该标的的持仓市值不超过总权益的指定比例

两个约束按顺序依次应用，最终买入量取所有约束中的最小值。

这是生产级策略推荐使用的 Entry Rule，因为它能防止单一标的过度集中。

In [6]:
from oxq.rules import SizedEntryRule

# 约束 1: 最多持有 200 股（请求买 500 股，被裁剪为 200）
rule_max = SizedEntryRule(signal="golden_cross", shares=500, max_position=200)
row = pd.Series({"close": 200.0, "golden_cross": True})
order = rule_max.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"max_position=200: 请求 500 股 → 实际买入 {order.shares} 股")

# 约束 2: 仓位不超过权益的 30%
# 权益 10 万 × 30% = 3 万，3 万 / 200 = 150 股
rule_pct = SizedEntryRule(signal="golden_cross", shares=1000, max_pct_equity=0.30)
order2 = rule_pct.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"max_pct_equity=0.30: 请求 1000 股 → 实际买入 {order2.shares} 股")

# 同时使用两个约束: 取更严格的那个
rule_both = SizedEntryRule(
    signal="golden_cross", shares=1000,
    max_position=200, max_pct_equity=0.30,
)
order3 = rule_both.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"两者叠加: 请求 1000 股 → 实际买入 {order3.shares} 股 (min(200, 150))")

max_position=200: 请求 500 股 → 实际买入 200 股
max_pct_equity=0.30: 请求 1000 股 → 实际买入 150 股
两者叠加: 请求 1000 股 → 实际买入 150 股 (min(200, 150))


### 1.5 四种 Entry Rule 回测对比

将四种 Entry Rule 放在同一套管道中回测，观察不同仓位控制方式对收益和风险的影响：

In [7]:
from oxq.rules import ExitRule

common = dict(
    universe=StaticUniverse((SYMBOL,)),
    indicators=COMMON_INDICATORS,
    signals=COMMON_SIGNALS,
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

entry_results = {}
for label, entry_rules in [
    ("固定100股", [EntryRule(signal="golden_cross", shares=100)]),
    ("全仓买入", [FullPositionEntryRule(signal="golden_cross")]),
    ("最多200股", [SizedEntryRule(signal="golden_cross", shares=500, max_position=200)]),
    ("最多30%权益", [SizedEntryRule(signal="golden_cross", shares=1000, max_pct_equity=0.30)]),
]:
    strategy = Strategy(name=label, entry_rules=entry_rules, **common)
    entry_results[label] = run(strategy)

compare(entry_results)

                        固定100股            全仓买入          最多200股         最多30%权益
------------------------------------------------------------------------------
          总收益率           4.42%          22.82%           8.84%           6.84%
        Sharpe            0.83            0.87            0.84            0.83
          最大回撤          -2.61%         -11.74%          -5.00%          -4.19%
          交易次数              13              13              13              13
          期末资产         104,421         122,819         108,842         106,836


**解读**：

- `EntryRule` 收益最低但最大回撤也最小——因为只用了很少的资金
- `FullPositionEntryRule` 收益最高但回撤也最大——全部资金暴露在市场风险中
- `SizedEntryRule` 在两者之间提供了灵活的仓位控制，适合生产环境

在实际使用中，`max_pct_equity` 尤其有用——它让仓位大小自动随账户规模缩放。

---
## 2. Exit Rules — 卖出规则

Exit Rules 在 Stage 4 执行，负责在条件满足时平仓。

`ExitRule` 是一个基于指标交叉的卖出规则：当快线低于慢线且持有仓位时，生成全仓 SELL 订单。

In [8]:
from oxq.rules import ExitRule

exit_rule = ExitRule(fast="sma_10", slow="sma_50")

# 有持仓 + 快线 < 慢线 → 卖出
row_exit = pd.Series({"close": 95.0, "sma_10": 97.0, "sma_50": 100.0})
portfolio_pos = Portfolio(
    cash=Decimal("50000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("102"))},
)
order = exit_rule.evaluate("AAPL", row_exit, portfolio_pos)
print(f"快线 < 慢线 + 有持仓: {order}")

# 无持仓 → 不操作
order2 = exit_rule.evaluate("AAPL", row_exit, Portfolio(cash=Decimal("100000")))
print(f"快线 < 慢线 + 无持仓: {order2}")

# 快线 > 慢线 → 不卖出
row_hold = pd.Series({"close": 105.0, "sma_10": 103.0, "sma_50": 100.0})
order3 = exit_rule.evaluate("AAPL", row_hold, portfolio_pos)
print(f"快线 > 慢线 + 有持仓: {order3}")

快线 < 慢线 + 有持仓: Order(symbol='AAPL', side='SELL', shares=100, order_type='market', limit_price=None, stop_price=None, trail_pct=None)
快线 < 慢线 + 无持仓: None
快线 > 慢线 + 有持仓: None


**与 Entry Rule 的执行顺序**：Engine 每个 bar 先执行 Exit（Stage 4），再执行 Entry（Stage 5）。
这意味着同一个 bar 可以先卖出再买入，但不会出现「买了就卖」的情况——因为 Entry 只在无持仓时触发。

> **注意**: `ExitRule` 使用指标列（`sma_10`, `sma_50`）判断退出条件，而非 Signal 列。
> 这是有意的设计——退出条件通常与入场条件不同。如果需要基于 Signal 退出，可以自定义 Rule。

---
## 3. Order Rules — 挂单规则（止损/止盈/追踪止损）

Order Rules 在 Stage 2b 执行。与 Entry/Exit Rule 生成的**市价单**不同，Order Rules 生成**条件挂单**——提交给 SimBroker 后，由 SimBroker 在后续每个 bar 检查是否触发。

### 工作流程

```
Order Rule       SimBroker.process_pending_orders()       成交
  ↓ 提交挂单            ↓ 每个 bar 检查                     ↓
stop SELL @140    bar close=150? 不触发                  ───
                  bar close=138? ≤140 → 触发!           Fill @140
```

### 自动去重

SimBroker 的 OrderBook 自动去重：同一 symbol + side + order_type 的新挂单会替换旧挂单。因此 Order Rule 可以每个 bar 都重新提交——它只是在更新止损/止盈价格，不会产生重复挂单。

open-xquant 提供 3 种 Order Rule：

| 规则 | 挂单类型 | 触发条件 | 成交价 |
|------|----------|----------|--------|
| `StopLossRule` | stop SELL | close ≤ stop_price | stop_price |
| `TakeProfitRule` | limit SELL | close ≥ limit_price | limit_price |
| `TrailingStopRule` | trailing_stop | close ≤ 高水位 × (1 - trail_pct) | stop_level |

### 3.1 StopLossRule — 固定止损

当持有仓位时，以 `avg_cost × (1 - threshold)` 为止损价，提交 stop SELL 挂单。

例如：买入均价 200，threshold=0.05 → 止损价 190。当收盘价跌至 190 或以下时，触发卖出。

In [9]:
from oxq.rules import StopLossRule

stop_loss = StopLossRule(threshold=0.05)  # 5% 止损

# 有持仓 → 生成止损挂单
row = pd.Series({"close": 200.0})
portfolio = Portfolio(
    cash=Decimal("80000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("200"))},
)
order = stop_loss.evaluate("AAPL", row, portfolio)
print(f"止损挂单: {order}")
print(f"  order_type = {order.order_type}")
print(f"  stop_price = {order.stop_price}  (avg_cost 200 × 0.95)")

# 无持仓 → 不生成
order2 = stop_loss.evaluate("AAPL", row, Portfolio(cash=Decimal("100000")))
print(f"\n无持仓: {order2}")

止损挂单: Order(symbol='AAPL', side='SELL', shares=100, order_type='stop', limit_price=None, stop_price=Decimal('190.00'), trail_pct=None)
  order_type = stop
  stop_price = 190.00  (avg_cost 200 × 0.95)

无持仓: None


### 3.2 TakeProfitRule — 固定止盈

当持有仓位时，以 `avg_cost × (1 + threshold)` 为止盈价，提交 limit SELL 挂单。

例如：买入均价 200，threshold=0.15 → 止盈价 230。当收盘价涨至 230 或以上时，触发卖出。

In [10]:
from oxq.rules import TakeProfitRule

take_profit = TakeProfitRule(threshold=0.15)  # 15% 止盈

order = take_profit.evaluate("AAPL", row, portfolio)
print(f"止盈挂单: {order}")
print(f"  order_type  = {order.order_type}")
print(f"  limit_price = {order.limit_price}  (avg_cost 200 × 1.15)")

止盈挂单: Order(symbol='AAPL', side='SELL', shares=100, order_type='limit', limit_price=Decimal('230.00'), stop_price=None, trail_pct=None)
  order_type  = limit
  limit_price = 230.00  (avg_cost 200 × 1.15)


### 3.3 TrailingStopRule — 追踪止损

生成 trailing_stop SELL 挂单。SimBroker 内部跟踪价格的**高水位 (HWM)**，当价格从高水位回撤超过 `trail_pct` 时触发。

```
价格走势:  100 → 110 → 105 → 103
HWM:      100 → 110    110    110
止损线(5%):        104.5  104.5  104.5
触发?:              否     否     是 (103 ≤ 104.5)
```

追踪止损的优势：在趋势上涨中，止损线随价格上移，锁住更多利润；在回调中自动触发，避免利润回吐。

In [11]:
from oxq.rules import TrailingStopRule

trailing = TrailingStopRule(trail_pct=0.05)  # 5% 追踪止损

order = trailing.evaluate("AAPL", row, portfolio)
print(f"追踪止损挂单: {order}")
print(f"  order_type = {order.order_type}")
print(f"  trail_pct  = {order.trail_pct}")
print()
print("注意: trail_pct 由 SimBroker 内部使用，Rule 只负责提交挂单。")
print("SimBroker 会自动跟踪高水位 (HWM) 并计算动态止损线。")

追踪止损挂单: Order(symbol='AAPL', side='SELL', shares=100, order_type='trailing_stop', limit_price=None, stop_price=None, trail_pct=0.05)
  order_type = trailing_stop
  trail_pct  = 0.05

注意: trail_pct 由 SimBroker 内部使用，Rule 只负责提交挂单。
SimBroker 会自动跟踪高水位 (HWM) 并计算动态止损线。


### 3.4 Order Rules 回测对比

在基础的 SMA 金叉/死叉策略上，分别加入不同的 Order Rule，观察对收益和风险的影响：

In [23]:
common_order = dict(
    universe=StaticUniverse((SYMBOL,)),
    indicators=COMMON_INDICATORS,
    signals=COMMON_SIGNALS,
    entry_rules=[EntryRule(signal="golden_cross", shares=100)],
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

order_results = {}
for label, order_rules in [
    ("无保护(基准)", []),
    ("5%止损", [StopLossRule(threshold=0.05)]),
    ("15%止盈", [TakeProfitRule(threshold=0.15)]),
    ("5%追踪止损", [TrailingStopRule(trail_pct=0.05)]),
    ("止损+止盈", [StopLossRule(threshold=0.05), TakeProfitRule(threshold=0.15)]),
    ("追踪止损+止盈", [TrailingStopRule(trail_pct=0.05), TakeProfitRule(threshold=0.15)]),
]:
    strategy = Strategy(name=label, order_rules=order_rules, **common_order)
    order_results[label] = run(strategy)

compare(order_results)

                       无保护(基准)            5%止损           15%止盈          5%追踪止损           止损+止盈         追踪止损+止盈
--------------------------------------------------------------------------------------------------------------
          总收益率           4.42%          38.74%          24.23%          22.01%          58.54%          63.95%
        Sharpe            0.83            1.10            0.82            0.83            1.29            1.28
          最大回撤          -2.61%          -1.97%          -1.66%          -2.24%          -1.43%          -1.66%
          交易次数              13              15              14              14              16              16
          期末资产         104,421         138,735         124,226         122,006         158,541         163,948


查看「止损+止盈」组合的交易记录，注意 `order_type` 列——可以看到 market、stop、limit 三种订单类型：

In [24]:
print("追踪止损+止盈 组合交易记录:")
show_trades(order_results["追踪止损+止盈"], max_rows=16)    


追踪止损+止盈 组合交易记录:
  日期                             方向     数量 订单类型                  成交价      手续费
  ----------------------------------------------------------------------------
  2023-10-17 00:00:00           BUY    100 market             175.10     0.00
  2023-10-23 00:00:00          SELL    100 market             171.00     0.00
  2023-11-10 00:00:00           BUY    100 market             184.48     0.00
  2024-01-09 00:00:00          SELL    100 market             183.24     0.00
  2024-01-30 00:00:00           BUY    100 market             186.11     0.00
  2024-01-31 00:00:00          SELL    100 trailing_stop      183.51     0.00
  2024-03-01 00:00:00          SELL    100 trailing_stop      178.09     0.00
  2024-05-06 00:00:00           BUY    100 market             180.07     0.00
  2024-06-12 00:00:00          SELL    100 limit              207.08     0.00
  2024-06-13 00:00:00          SELL    100 limit              207.08     0.00
  2024-07-24 00:00:00          SELL    100 trai

**解读**：

- `market` 订单来自 EntryRule（买入）和 ExitRule（死叉卖出）
- `stop` 订单来自 StopLossRule 在 SimBroker 中触发
- `limit` 订单来自 TakeProfitRule 在 SimBroker 中触发
- 止损和止盈互不干扰——SimBroker 为每种 order_type 维护独立的挂单
- 止损+止盈组合通常比单独使用效果更好，因为它同时控制了下行风险和锁定了上行利润

---
## 4. Risk Rules — 组合级熔断保护

Risk Rules 在 Stage 1（**最先**）执行。它们是策略的「安全网」——在极端行情下自动冻结交易，防止不可控的损失。

### 与其他 Rule 的关键区别

| 维度 | Entry/Exit/Order/Rebalance Rule | Risk Rule |
|------|--------------------------------|----------|
| 返回值 | `Order \| None` | `tuple[Order \| None, bool]` |
| hold 信号 | 无 | 可冻结后续所有阶段 |
| 执行阶段 | Stage 2b~5 | **Stage 1**（最先） |
| 作用范围 | 单个标的 | 组合级别 |

Risk Rule 返回的 `bool` 就是 hold 信号：当任一 Risk Rule 返回 `hold=True` 时，Engine 跳过 Stage 2b~5（不生成新订单）。但 Stage 2a（`process_pending_orders`）不受影响——已经挂出的止损单仍然可以触发。

open-xquant 提供 2 种 Risk Rule：

| 规则 | 监控指标 | 触发行为 |
|------|----------|----------|
| `MaxDrawdownRisk` | 组合峰值到谷值回撤 | 清仓 + 冻结 |
| `DailyLossLimitRisk` | 单日亏损 | 仅冻结（不清仓） |

### 4.1 MaxDrawdownRisk — 最大回撤熔断

跟踪组合的历史峰值。当「(峰值 - 当前值) / 峰值」超过 `max_drawdown` 时：

1. 对当前有持仓的标的生成 SELL 订单（清仓）
2. 返回 `hold=True` 冻结后续所有阶段

这是最激进的保护措施——一旦触发，不仅阻止新交易，还会清空现有持仓。

In [14]:
from oxq.rules import MaxDrawdownRisk

risk = MaxDrawdownRisk(max_drawdown=0.15)  # 15% 回撤熔断

portfolio = Portfolio(
    cash=Decimal("0"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("150"))},
)

# 第一次评估: 组合值 = 100 × 200 = 20000，设为峰值
row_high = pd.Series({"close": 200.0})
order, hold = risk.evaluate("AAPL", row_high, portfolio)
print(f"价格 200 (峰值): order={order}, hold={hold}")

# 第二次评估: 组合值 = 100 × 160 = 16000，回撤 = 4000/20000 = 20% > 15%
row_low = pd.Series({"close": 160.0})
order, hold = risk.evaluate("AAPL", row_low, portfolio)
print(f"价格 160 (回撤20%): order={order}, hold={hold}")
if order:
    print(f"  → 清仓订单: SELL {order.shares} 股")

价格 200 (峰值): order=None, hold=False
价格 160 (回撤20%): order=Order(symbol='AAPL', side='SELL', shares=100, order_type='market', limit_price=None, stop_price=None, trail_pct=None), hold=True
  → 清仓订单: SELL 100 股


### 4.2 DailyLossLimitRisk — 单日亏损熔断

记录每个交易日开始时的组合价值。当日内亏损超过 `max_daily_loss` 时，返回 `hold=True` 冻结后续阶段。

与 `MaxDrawdownRisk` 的区别：
- **不清仓** — 只冻结新订单生成
- **次日自动恢复** — 新的交易日会重置基准值
- 适合日内波动剧烈但长期趋势明确的行情

In [15]:
from oxq.rules import DailyLossLimitRisk

daily_risk = DailyLossLimitRisk(max_daily_loss=0.03)  # 3% 单日亏损限制

portfolio = Portfolio(
    cash=Decimal("0"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("150"))},
)

# 日初评估: 记录基准值 = 100 × 200 = 20000
row1 = pd.Series({"close": 200.0}, name=pd.Timestamp("2024-01-02"))
order, hold = daily_risk.evaluate("AAPL", row1, portfolio)
print(f"日初 价格 200: hold={hold}  (记录基准值 20000)")

# 同日价格下跌: 100 × 190 = 19000, 日损 = 1000/20000 = 5% > 3%
row2 = pd.Series({"close": 190.0}, name=pd.Timestamp("2024-01-02"))  # 注意: 同一天
order, hold = daily_risk.evaluate("AAPL", row2, portfolio)
print(f"同日 价格 190: hold={hold}  (日损 5% > 3%, 冻结交易)")
print(f"  → order={order}  (DailyLossLimitRisk 不清仓)")

日初 价格 200: hold=False  (记录基准值 20000)
同日 价格 190: hold=True  (日损 5% > 3%, 冻结交易)
  → order=None  (DailyLossLimitRisk 不清仓)


### 4.3 Risk Rules 回测对比

In [16]:
common_risk = dict(
    universe=StaticUniverse((SYMBOL,)),
    indicators=COMMON_INDICATORS,
    signals=COMMON_SIGNALS,
    entry_rules=[FullPositionEntryRule(signal="golden_cross")],
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

risk_results = {}
for label, risk_rules in [
    ("无保护(基准)", []),
    ("15%最大回撤", [MaxDrawdownRisk(max_drawdown=0.15)]),
    ("3%单日亏损", [DailyLossLimitRisk(max_daily_loss=0.03)]),
    ("双重保护", [
        MaxDrawdownRisk(max_drawdown=0.15),
        DailyLossLimitRisk(max_daily_loss=0.03),
    ]),
]:
    strategy = Strategy(name=label, risk_rules=risk_rules, **common_risk)
    risk_results[label] = run(strategy)

compare(risk_results)

                       无保护(基准)         15%最大回撤          3%单日亏损            双重保护
------------------------------------------------------------------------------
          总收益率          22.82%          22.82%          22.82%          22.82%
        Sharpe            0.87            0.87            0.87            0.87
          最大回撤         -11.74%         -11.74%         -11.74%         -11.74%
          交易次数              13              13              13              13
          期末资产         122,819         122,819         122,819         122,819


**解读**：

- Risk Rules 是「保险」——在行情温和时不影响策略表现，但在极端行情下能有效控制损失
- 本例中 AAPL 2023-2024 走势较好，最大回撤 ~12%，未触发 15% 的熔断线
- 在更剧烈的行情或更敏感的阈值下，Risk Rules 的保护效果会更明显
- `DailyLossLimitRisk` 的冻结是日内的——次日自动恢复交易
- 两者可叠加使用：任一触发 `hold=True` 即冻结后续阶段

---
## 5. Rebalance Rules — 再平衡规则

Rebalance Rules 在 Stage 3 执行。`RebalanceRule` 每隔 `frequency` 个 bar，读取 Signal 层生成的权重列，计算目标持仓与当前持仓的差值，生成 BUY 或 SELL 订单使组合向目标权重收敛。

适用于：动量轮动、等权再平衡、风险平价等需要定期调仓的策略。

```
Signal 输出权重    RebalanceRule 计算差值      生成订单
AAPL: 0.5          目标 250 股, 当前 100 股    BUY 150
MSFT: 0.3          目标 150 股, 当前 200 股    SELL 50
GOOGL: 0.2         目标 100 股, 当前 0 股      BUY 100
```

In [17]:
from oxq.rules import RebalanceRule

rebalance = RebalanceRule(weight_col="target_weight", frequency=10)

# 模拟: 目标权重 0.5, 当前无持仓, 组合总值 10 万
row = pd.Series({"close": 200.0, "target_weight": 0.5}, name=pd.Timestamp("2024-01-01"))
portfolio = Portfolio(cash=Decimal("100000"))

# 注意: RebalanceRule 内部有 bar 计数器，只在第 frequency 个 bar 才执行
# 前 9 个 bar 返回 None，第 10 个 bar 才生成订单
for i in range(10):
    row_i = pd.Series(
        {"close": 200.0, "target_weight": 0.5},
        name=pd.Timestamp(f"2024-01-{i+1:02d}"),
    )
    order = rebalance.evaluate("AAPL", row_i, portfolio)
    if order:
        print(f"Bar {i+1}: 生成订单 → {order.side} {order.shares} 股")
    else:
        print(f"Bar {i+1}: 不是再平衡日")

print()
print("说明: frequency=10 表示每 10 个 bar 再平衡一次")
print("目标: 组合 10 万 × 50% = 5 万 → 5 万 / 200 = 250 股")

Bar 1: 不是再平衡日
Bar 2: 不是再平衡日
Bar 3: 不是再平衡日
Bar 4: 不是再平衡日
Bar 5: 不是再平衡日
Bar 6: 不是再平衡日
Bar 7: 不是再平衡日
Bar 8: 不是再平衡日
Bar 9: 不是再平衡日
Bar 10: 生成订单 → BUY 250 股

说明: frequency=10 表示每 10 个 bar 再平衡一次
目标: 组合 10 万 × 50% = 5 万 → 5 万 / 200 = 250 股


> **提示**: RebalanceRule 通常与 `TopNRanking`、`EqualWeight` 等 Signal 配合使用。
> 这些 Signal 输出每个标的的目标权重列，RebalanceRule 据此计算调仓订单。
> 完整示例参见 `examples/tutorials/rotation_strategy.ipynb`。

---
## 6. 交易成本 — 手续费与滑点

交易成本不是 Rule，但它直接影响 Rule 产生的订单的最终效果。SimBroker 通过两个 Protocol 接口模拟交易成本：

| 模型 | 作用 | 公式 |
|------|------|------|
| `PercentageFee` | 手续费 | `max(fill_price × shares × rate, min_fee)` |
| `PercentageSlippage` | 滑点 | BUY: `price × (1 + rate)`，SELL: `price × (1 - rate)` |

**执行顺序**：先计算滑点调整后的成交价，再按调整后价格计算手续费。

手续费记录在 `Fill.fee` 上，由 Engine 在 `_apply_fill` 时从 `portfolio.cash` 中扣除。

In [18]:
from oxq.trade import PercentageFee, PercentageSlippage

# 查看手续费模型的默认参数
fee = PercentageFee()  # rate=0.1%, min_fee=5
print(f"PercentageFee: rate={fee.rate}, min_fee={fee.min_fee}")

# 查看滑点模型的默认参数
slip = PercentageSlippage()  # rate=0.1%
print(f"PercentageSlippage: rate={slip.rate}")

# 手动计算: 买 100 股 AAPL @200
from oxq.core.types import Order
order = Order(symbol="AAPL", side="BUY", shares=100)

raw_price = Decimal("200")
slipped_price = slip.adjust(order, raw_price)
fee_amount = fee.calculate(order, slipped_price)

print(f"\n手动计算: BUY 100 股 @200")
print(f"  滑点后价格: {slipped_price}  (200 × 1.001)")
print(f"  手续费:     {fee_amount}  ({slipped_price} × 100 × 0.001)")
print(f"  总成本:     {slipped_price * 100 + fee_amount}")

PercentageFee: rate=0.001, min_fee=5
PercentageSlippage: rate=0.001

手动计算: BUY 100 股 @200
  滑点后价格: 200.200  (200 × 1.001)
  手续费:     20.020000  (200.200 × 100 × 0.001)
  总成本:     20040.020000


### 交易成本对策略的影响

同一个策略，在不同交易成本下的表现差异：

In [19]:
strategy = Strategy(
    name="cost_test",
    universe=StaticUniverse((SYMBOL,)),
    indicators=COMMON_INDICATORS,
    signals=COMMON_SIGNALS,
    entry_rules=[EntryRule(signal="golden_cross", shares=100)],
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

cost_configs = {
    "无成本(基准)": {},
    "0.1%手续费": {
        "fee_model": PercentageFee(rate=Decimal("0.001"), min_fee=Decimal("5")),
    },
    "0.1%滑点": {
        "slippage_model": PercentageSlippage(rate=Decimal("0.001")),
    },
    "费用+滑点": {
        "fee_model": PercentageFee(rate=Decimal("0.001"), min_fee=Decimal("5")),
        "slippage_model": PercentageSlippage(rate=Decimal("0.001")),
    },
    "高成本": {
        "fee_model": PercentageFee(rate=Decimal("0.003"), min_fee=Decimal("10")),
        "slippage_model": PercentageSlippage(rate=Decimal("0.005")),
    },
}

cost_results = {}
for label, kwargs in cost_configs.items():
    cost_results[label] = run(strategy, **kwargs)

compare(cost_results, extra_rows=[
    ("总手续费", lambda r: f"{sum(f.fee for f in r.trades):.0f}"),
])

                       无成本(基准)         0.1%手续费          0.1%滑点           费用+滑点             高成本
----------------------------------------------------------------------------------------------
          总收益率           4.42%           4.16%           4.16%           3.90%           2.34%
        Sharpe            0.83            0.78            0.78            0.73            0.44
          最大回撤          -2.61%          -2.62%          -2.62%          -2.62%          -3.18%
          交易次数              13              13              13              13              13
          期末资产         104,421         104,160         104,160         103,900         102,335
          总手续费               0             261               0             261             782


**解读**：

- 交易成本是回测与实盘表现差异的主要来源之一
- 对低频策略（本例 ~13 笔交易），0.1% 的费率影响约 0.3%~0.5% 收益率
- 对高频策略，影响会成倍放大
- `min_fee` 对小额交易的影响尤其大——100 股 × 200 = 20,000 的交易，0.1% 费率只有 20 元，但 min_fee=5 兜底
- 在策略开发后期，加入交易成本是评估可行性的必要步骤

---
## 7. 完整策略 — 所有 Rule 协同工作

最后，将所有 Rule 组合成一个完整的、具备风控能力的策略：

```
每个 bar 的执行顺序:

1. MaxDrawdownRisk   → 检查回撤，超过 15% 则清仓 + 冻结
                        ↓ (如果冻结，跳过 2b~5)
2a. SimBroker        → 检查挂单：5% 止损单或 20% 止盈单是否触发？
2b. StopLossRule     → 提交/更新止损挂单 (stop_price = avg_cost × 0.95)
    TakeProfitRule   → 提交/更新止盈挂单 (limit_price = avg_cost × 1.20)
3. (无 Rebalance)    → 跳过
4. ExitRule          → 死叉时全仓卖出
5. SizedEntryRule    → 金叉时买入，仓位不超过权益的 50%
```

In [20]:
full_strategy = Strategy(
    name="full_rules_demo",
    hypothesis=(
        "SMA 金叉买入 + 死叉卖出，配合止损止盈和回撤熔断，"
        "在控制下行风险的同时捕获趋势收益"
    ),
    objectives={
        "total_return": {"min": 0.0},
        "sharpe_ratio": {"min": 0.3},
        "max_drawdown": {"max": -0.15},
    },
    universe=StaticUniverse((SYMBOL,)),
    indicators=COMMON_INDICATORS,
    signals=COMMON_SIGNALS,
    risk_rules=[
        MaxDrawdownRisk(max_drawdown=0.15),
    ],
    order_rules=[
        StopLossRule(threshold=0.05),
        TakeProfitRule(threshold=0.20),
    ],
    entry_rules=[
        SizedEntryRule(signal="golden_cross", shares=500, max_pct_equity=0.50),
    ],
    exit_rules=[
        ExitRule(fast="sma_10", slow="sma_50"),
    ],
)

result = run(
    full_strategy,
    fee_model=PercentageFee(rate=Decimal("0.001"), min_fee=Decimal("5")),
    slippage_model=PercentageSlippage(rate=Decimal("0.001")),
)

print(f"总收益率:   {result.total_return():.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"最大回撤:   {result.max_drawdown():.2%}")
print(f"交易次数:   {len(result.trades)}")
print(f"期末资产:   {result.equity_curve[-1][1]:,.0f}")
print(f"总手续费:   {sum(f.fee for f in result.trades):.2f}")

总收益率:   245.35%
Sharpe Ratio: 1.30
最大回撤:   -4.50%
交易次数:   16
期末资产:   345,348
总手续费:   1374.35


### 目标检查

In [21]:
metrics = {
    "total_return": result.total_return(),
    "sharpe_ratio": result.sharpe_ratio(),
    "max_drawdown": result.max_drawdown(),
}

print("目标检查:")
for metric_name, bounds in full_strategy.objectives.items():
    actual = metrics[metric_name]
    passed = True
    if "min" in bounds:
        passed = passed and actual >= bounds["min"]
    if "max" in bounds:
        passed = passed and actual <= bounds["max"]
    status = "PASS" if passed else "FAIL"
    print(f"  {metric_name:<16} = {actual:>8.4f}  [{status}]")

目标检查:
  total_return     =   2.4535  [PASS]
  sharpe_ratio     =   1.2963  [PASS]
  max_drawdown     =  -0.0450  [FAIL]


### 完整交易记录

In [22]:
show_trades(result, max_rows=20)

  日期                             方向     数量 订单类型                  成交价      手续费
  ----------------------------------------------------------------------------
  2023-10-17 00:00:00           BUY    285 market             175.27    49.95
  2023-10-23 00:00:00          SELL    285 market             170.82    48.69
  2023-10-26 00:00:00          SELL    285 stop               166.34    47.41
  2023-11-10 00:00:00           BUY    395 market             184.67    72.94
  2024-01-09 00:00:00          SELL    395 market             183.05    72.31
  2024-01-30 00:00:00           BUY    390 market             186.29    72.65
  2024-02-05 00:00:00          SELL    390 market             185.56    72.37
  2024-03-04 00:00:00          SELL    390 stop               176.80    68.95
  2024-05-06 00:00:00           BUY    500 market             180.25    90.13
  2024-07-02 00:00:00          SELL    500 limit              216.30   108.15
  2024-07-03 00:00:00          SELL    500 limit              2

---
## 小结

### 5 类 Rule 速查表

| 阶段 | Rule 类型 | 内置实现 | 返回值 | 职责 |
|------|-----------|---------|--------|------|
| Stage 1 | Risk Rules | `MaxDrawdownRisk`, `DailyLossLimitRisk` | `tuple[Order\|None, bool]` | 组合级熔断，可冻结后续阶段 |
| Stage 2b | Order Rules | `StopLossRule`, `TakeProfitRule`, `TrailingStopRule` | `Order\|None` | 提交条件挂单（止损/止盈/追踪） |
| Stage 3 | Rebalance Rules | `RebalanceRule` | `Order\|None` | 定期再平衡至目标权重 |
| Stage 4 | Exit Rules | `ExitRule` | `Order\|None` | 信号/条件触发平仓 |
| Stage 5 | Entry Rules | `EntryRule`, `TargetValueEntryRule`, `FullPositionEntryRule`, `SizedEntryRule` | `Order\|None` | 信号触发建仓 |

### 交易成本

| 模型 | 实现 | 作用 |
|------|------|------|
| `FeeModel` | `PercentageFee` | 按交易额比例收费，有最低收费 |
| `SlippageModel` | `PercentageSlippage` | 模拟市场冲击（买贵卖便宜） |

### 仓位控制

| 函数/参数 | 用法 |
|----------|------|
| `clip_to_max_position` | 限制总持仓股数 |
| `clip_to_pct_equity` | 限制持仓市值占比 |
| `SizedEntryRule.max_position` | Entry Rule 内置的股数上限 |
| `SizedEntryRule.max_pct_equity` | Entry Rule 内置的权益比例上限 |

### 核心设计原则

- **声明式组合** — 所有 Rule 通过 `Strategy` dataclass 组合，不修改 Engine 代码
- **固定执行顺序** — Risk → Order → Rebalance → Exit → Entry，保证行为可预测
- **hold 机制** — Risk Rules 可冻结后续阶段，但不影响已挂出的条件单
- **自动去重** — SimBroker 的 OrderBook 确保同类挂单不重复
- **Protocol 接口** — FeeModel、SlippageModel 可自定义实现

### 相关教程

- `engine_module.ipynb` — Engine 四阶段管道详解
- `signal_comparison.ipynb` — Signal 模块详解（Crossover, TopNRanking, EqualWeight 等）
- `rotation_strategy.ipynb` — RebalanceRule + TopNRanking 动量轮动策略完整示例